# 1. Descomposición de Tareas (Planificación)

## 1.1. Contexto del Problema

La empresa de telecomunicaciones CallMeMaybe enfrenta una alta tasa de llamadas perdidas y tiempos de espera prolongados, lo que afecta la satisfacción del cliente y la eficiencia operativa. El equipo directivo necesita identificar a los operadores con bajo desempeño para implementar acciones correctivas.

## 1.2. Objetivos del Análisis

### **Objetivo principal:** Desarrollar un modelo de evaluación que permita clasificar a los operadores según su nivel de eficiencia.

### **Objetivos específicos:**

- Determinar métricas clave de eficiencia: tasa de llamadas perdidas (entrantes), tiempo de espera promedio y volumen de llamadas atendidas.
- Identificar patrones de ineficacia (por perfil: solo entrante, solo saliente, mixto).
- Proponer recomendaciones basadas en datos para mejorar la gestión del centro de llamadas.

## 1.3. Pasos de la descomposición

**1. Carga y exploración inicial de datos**
- Importar telecom_dataset_new.csv y telecom_clients.csv.
- Revisar estructura, tipos de datos y calidad (nulos, duplicados, valores atípicos).

**2.Preprocesamiento y limpieza**
- Eliminar duplicados exactos.
- Convertir fechas a tipo datetime y campos de identificación a string.
- Manejar valores ausentes en operator_id: etiquetar como 'unknown'.
- Corregir inconsistencias lógicas (llamadas perdidas con duración > 0, llamadas atendidas sin operador).

**3.Análisis exploratorio de datos (EDA)**
- Analizar la distribución de la duración de llamadas y detectar outliers.
- Evaluar el tiempo de espera (wait_time) y su relación con la tasa de pérdida.
- Segmentar por dirección (in/out), tipo de llamada (interna/externa) y día de la semana.
- Identificar operadores con mayor carga de trabajo y anomalías.

**4. Definición de criterios de eficiencia**
- Establecer umbrales para considerar a un operador como ineficaz:
  - Solo entrante: alta tasa de pérdida (> mediana del equipo) y alto tiempo de espera (> 60 segundos).
  - Solo saliente: muy bajo volumen de llamadas (percentil 25 del equipo).
  - Mixto: combina ambos criterios.

- Clasificar también operadores en estado crítico (severamente ineficaces).

**5. Pruebas de hipótesis**
- Correlacionar tiempo de espera con llamadas perdidas.
- Comparar duración de llamadas externas vs internas.
- Los operadores con mayor volumen de llamadas salientes tienen menor tasa de pérdida entrante
- El tiempo de espera promedio es mayor en operadores mixtos que en operadores solo entrantes
- Existe relación entre el tiempo de espera y la duración de la llamada (solo llamadas atendidas)

**6. Conclusiones y recomendaciones**
Resumir hallazgos clave, identificar operadores específicos a intervenir y proponer acciones de mejora.

## 1.4. Identificación del Problema de Negocio (Ampliación)

El sector de telecomunicaciones es altamente competitivo y la experiencia del cliente es un factor clave de diferenciación. La empresa "CallMeMaybe" busca optimizar su centro de llamadas para reducir la frustración del usuario y maximizar la eficiencia de sus recursos.

Valor para el negocio:

Realizar este análisis permitirá:

- Reducir la tasa de abandono de clientes: Un cliente que espera mucho o no es atendido tiene una alta probabilidad de cambiar de operador.
- Optimizar la asignación de personal: Identificar operadores con bajo rendimiento permite redirigir recursos (capacitación, supervisión o ajuste de horarios).
- Disminuir costos operativos: Al detectar ineficiencias (como llamadas internas excesivamente largas o volúmenes anormalmente bajos), se puede mejorar la productividad general.
- Mejorar la trazabilidad: El análisis revela problemas de registro (operadores unknown), lo que impulsa a corregir fallos en los sistemas de logging.

## 1.5. Descripción de los Datos

Trabajaremos con dos archivos principales:

**A. `telecom_dataset_new.csv`**
- **Volumen:** 53,902 registros y 9 columnas.
- **Variables principales:**
  - `user_id`: Identificador del cliente (entero).
  - `date`: Fecha y hora de la llamada (objeto, requiere conversión a datetime).
  - `direction`: Dirección de la llamada (`in` = entrante, `out` = saliente).
  - `internal`: Indica si es una llamada interna entre operadores (booleano).
  - `operator_id`: Identificador del operador (flotante, con valores nulos).
  - `is_missed_call`: Indica si la llamada fue perdida (booleano).
  - `calls_count`: Número de llamadas agrupadas en el registro.
  - `call_duration`: Duración de la conversación (segundos).
  - `total_call_duration`: Duración total incluyendo tiempo de espera (segundos).
- **Calidad de los datos:**
  - **Valores nulos:** La columna `operator_id` tiene ~15% de nulos (8,289 valores). La columna `internal` tiene un porcentaje menor (0.2%).
  - **Duplicados:** Se detectaron 4,900 registros duplicados exactos, que deben eliminarse.
  - **Tipos incorrectos:** `date` es texto, debe convertirse a datetime; `operator_id` y `user_id` deben ser strings para su correcta identificación.

**B. `telecom_clients.csv`**
- **Volumen:** 732 registros y 3 columnas.
- **Variables:**
  - `user_id`: Identificador del cliente (entero).
  - `tariff_plan`: Plan tarifario (A, B o C).
  - `date_start`: Fecha de inicio del contrato (objeto, requiere conversión).
- **Calidad de los datos:** No presenta valores nulos ni duplicados.

**Variables de mayor interés para el análisis:**
- `operator_id`: Nos permitirá agregar el desempeño por operador.
- `direction`: Para separar la eficiencia en atención al cliente (`in`) de la productividad en llamadas salientes (`out`).
- `is_missed_call`: Indicador principal de ineficacia en llamadas entrantes.
- `wait_time` (a calcular): `total_call_duration - call_duration`, métrica clave para el servicio al cliente.
- `internal`: Para analizar si el tiempo entre operadores resta eficiencia.

---

## 1.6. Hipótesis a Validar

| Hipótesis | Descripción | Hipótesis Nula (H0) | Hipótesis Alternativa (H1) |
| :--- | :--- | :--- | :--- |
| **H1** | Existe una correlación positiva entre el tiempo de espera (`wait_time`) y la probabilidad de que una llamada entrante sea perdida. | No existe correlación entre el tiempo de espera y la tasa de pérdida. | A mayor tiempo de espera, mayor es la tasa de llamadas perdidas. |
| **H2** | Las llamadas externas (a clientes) tienen una duración significativamente mayor que las llamadas internas (entre operadores). | No hay diferencia significativa en la duración media entre llamadas externas e internas. | La duración media de las llamadas externas es significativamente mayor que la de las internas. |
| **H3** | Los operadores con mayor volumen de llamadas salientes tienen menor tasa de pérdida de llamadas entrantes. | No existe relación entre el volumen de llamadas salientes y la tasa de pérdida entrante. | A mayor volumen de llamadas salientes, menor es la tasa de pérdida de llamadas entrantes. |
| **H4** | El tiempo de espera promedio es mayor en operadores mixtos (que atienden tanto llamadas entrantes como salientes) que en operadores dedicados solo a llamadas entrantes. | No hay diferencia significativa en el tiempo de espera promedio entre operadores mixtos y operadores solo entrantes. | El tiempo de espera promedio es significativamente mayor en operadores mixtos. |
| **H5** | Existe una relación entre el tiempo de espera y la duración de la llamada, considerando únicamente las llamadas atendidas. | No existe correlación entre el tiempo de espera y la duración de la llamada atendida. | Existe una correlación significativa entre el tiempo de espera y la duración de la llamada atendida. |
---

## 1.7. Indicadores Clave (KPIs)

| KPI | Definición | Fórmula / Cálculo | Unidad | Interpretación |
| :--- | :--- | :--- | :--- | :--- |
| **Tasa de llamadas perdidas (entrantes)** | Porcentaje de llamadas entrantes que no fueron atendidas. | (Nº llamadas `in` perdidas / Total llamadas `in`) * 100 | Porcentaje (%) | Mide la calidad del servicio al cliente. Un valor alto indica insuficiencia de personal o mala asignación. |
| **Tiempo de espera promedio (entrantes)** | Tiempo medio que un cliente espera antes de ser atendido. | `mean(total_call_duration - call_duration)` para llamadas `in` atendidas. | Segundos | Refleja la eficiencia en la atención. Idealmente debe ser < 60 segundos. |
| **Volumen de llamadas salientes** | Número total de llamadas realizadas por un operador hacia el exterior. | `sum(calls_count)` para `direction = 'out'` por operador. | Conteo | Mide la proactividad y productividad del operador en campañas. Un volumen muy bajo puede indicar inactividad. |
| **Perfil de ineficacia** | Clasificación del operador según su patrón de bajo desempeño. | `'SOLO ENTRANTE'`, `'SOLO SALIENTE'` o `'MIXTO'` | Categórico | Permite segmentar las acciones correctivas (capacitación en atención vs. incentivos por volumen). |
| **Trazabilidad del operador** | Porcentaje de llamadas que tienen un `operator_id` válido. | (Registros con operator_id válido / Total registros) * 100 | Porcentaje (%) | Control de calidad del dato. Un valor bajo (< 90%) indica problemas en el sistema de logging. |
